In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import pandas as pd

file_path = "/content/drive/MyDrive/Dude_Wheres_My_Bike_Project/01_Data/raw/datarawFahrraddiebstahl_raw_2026-08-10.csv"

df_raw = pd.read_csv(
    file_path,
    encoding="latin1",
    dtype={"LOR": "string"}
)

print("Rows and columns:", df_raw.shape)

df_raw.head()



Rows and columns: (26965, 11)


,ANGELEGT_AM,TATZEIT_ANFANG_DATUM,TATZEIT_ANFANG_STUNDE,TATZEIT_ENDE_DATUM,TATZEIT_ENDE_STUNDE,LOR,SCHADENSHOEHE,VERSUCH,ART_DES_FAHRRADS,DELIKT,ERFASSUNGSGRUND
0,08.08.2026,08.08.2026,0,08.08.2026,0,01200521,0,Ja,Rennrad,Fahrraddiebstahl,Sonstiger schwerer Diebstahl von Fahrrädern
1,08.08.2026,01.08.2026,11,01.08.2026,13,03601347,2150,Nein,Citybike,Fahrraddiebstahl,Sonstiger schwerer Diebstahl von Fahrrädern
2,08.08.2026,08.08.2026,0,08.08.2026,0,01200521,0,Ja,Rennrad,Fahrraddiebstahl,Sonstiger schwerer Diebstahl von Fahrrädern
3,08.08.2026,05.08.2026,11,05.08.2026,17,03701554,2650,Nein,Herrenfahrrad,Fahrraddiebstahl,Sonstiger schwerer Diebstahl von Fahrrädern
4,08.08.2026,08.08.2026,12,08.08.2026,17,11401034,1499,Nein,Rennrad,Fahrraddiebstahl,Sonstiger schwerer Diebstahl von Fahrrädern


In [3]:
# checking the column names and data types.
print("Column names:")
print(df_raw.columns.tolist())

print("\nDataset information:")
df_raw.info()

Column names:
['ANGELEGT_AM', 'TATZEIT_ANFANG_DATUM', 'TATZEIT_ANFANG_STUNDE', 'TATZEIT_ENDE_DATUM', 'TATZEIT_ENDE_STUNDE', 'LOR', 'SCHADENSHOEHE', 'VERSUCH', 'ART_DES_FAHRRADS', 'DELIKT', 'ERFASSUNGSGRUND']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26965 entries, 0 to 26964
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   ANGELEGT_AM            26965 non-null  object
 1   TATZEIT_ANFANG_DATUM   26965 non-null  object
 2   TATZEIT_ANFANG_STUNDE  26965 non-null  int64 
 3   TATZEIT_ENDE_DATUM     26965 non-null  object
 4   TATZEIT_ENDE_STUNDE    26965 non-null  int64 
 5   LOR                    26965 non-null  string
 6   SCHADENSHOEHE          26965 non-null  int64 
 7   VERSUCH                26965 non-null  object
 8   ART_DES_FAHRRADS       26965 non-null  object
 9   DELIKT                 26965 non-null  object
 10  ERFASSUNGSGRUND        26965 non-null  objec

In [4]:
# checking missing values and duplicate-looking rows
missing_values = df_raw.isna().sum()

print("Missing values by column:")
print(missing_values)

duplicate_count = df_raw.duplicated().sum()

print("\nDuplicate-looking rows:", duplicate_count)

Missing values by column:
ANGELEGT_AM              0
TATZEIT_ANFANG_DATUM     0
TATZEIT_ANFANG_STUNDE    0
TATZEIT_ENDE_DATUM       0
TATZEIT_ENDE_STUNDE      0
LOR                      0
SCHADENSHOEHE            0
VERSUCH                  0
ART_DES_FAHRRADS         0
DELIKT                   0
ERFASSUNGSGRUND          0
dtype: int64

Duplicate-looking rows: 37


### Initial data-quality findings

- The raw dataset contains 26,965 rows and 11 columns.
- No missing values were detected.
- 37 rows have identical values across all available columns.
- These rows will not be removed without further evidence because the dataset does not contain a unique incident ID.
- The raw dataset remains unchanged.

In [5]:
# converting the date columns in a working copy
# Create a working copy and convert date columns

df = df_raw.copy()

date_columns = [
    "ANGELEGT_AM",
    "TATZEIT_ANFANG_DATUM",
    "TATZEIT_ENDE_DATUM"
]

for column in date_columns:
    df[column] = pd.to_datetime(
        df[column],
        format="%d.%m.%Y",
        errors="coerce"
    )

date_summary = pd.DataFrame({
    "earliest_date": df[date_columns].min(),
    "latest_date": df[date_columns].max(),
    "invalid_dates": df[date_columns].isna().sum()
})

display(date_summary)

,earliest_date,latest_date,invalid_dates
ANGELEGT_AM,2025-01-01,2026-08-08,0
TATZEIT_ANFANG_DATUM,2025-01-01,2026-08-08,0
TATZEIT_ENDE_DATUM,2025-01-01,2026-08-08,0


In [6]:
# validating the hour columns and LOR codes
# Validate hours and LOR codes

hour_columns = [
    "TATZEIT_ANFANG_STUNDE",
    "TATZEIT_ENDE_STUNDE"
]

for column in hour_columns:
    print(f"{column}:")
    print("Minimum:", df[column].min())
    print("Maximum:", df[column].max())
    print("Outside 0–23:", (~df[column].between(0, 23)).sum())
    print()

print("LOR code lengths:")
print(df["LOR"].str.len().value_counts().sort_index())

invalid_lor = ~df["LOR"].str.fullmatch(r"\d{8}")

print("\nInvalid LOR codes:", invalid_lor.sum())

TATZEIT_ANFANG_STUNDE:
Minimum: 0
Maximum: 23
Outside 0–23: 0

TATZEIT_ENDE_STUNDE:
Minimum: 0
Maximum: 23
Outside 0–23: 0

LOR code lengths:
LOR
8    26965
Name: count, dtype: Int64

Invalid LOR codes: 0


In [7]:
# inspecting the categorical columns

categorical_columns = [
    "VERSUCH",
    "ART_DES_FAHRRADS",
    "DELIKT",
    "ERFASSUNGSGRUND"
]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print("Unique values:", df[column].nunique())
    print(df[column].value_counts(dropna=False))


--- VERSUCH ---
Unique values: 3
VERSUCH
Nein         26804
Ja             154
Unbekannt        7
Name: count, dtype: int64

--- ART_DES_FAHRRADS ---
Unique values: 13
ART_DES_FAHRRADS
Herrenfahrrad        10914
Damenfahrrad          5926
Fahrrad               4719
Mountainbike          1325
Kinderfahrrad         1208
Rennrad                923
diverse Fahrräder      653
Trekkingrad            518
Citybike               511
Lastenfahrrad          137
Hollandrad              77
Klapprad                51
BMX                      3
Name: count, dtype: int64

--- DELIKT ---
Unique values: 2
DELIKT
Fahrraddiebstahl             25614
Keller- und Bodeneinbruch     1351
Name: count, dtype: int64

--- ERFASSUNGSGRUND ---
Unique values: 4
ERFASSUNGSGRUND
Sonstiger schwerer Diebstahl von Fahrrädern                        24092
Einfacher Diebstahl von Fahrrädern                                  1452
Sonstiger schwerer Diebstahl in/aus Keller/Boden von Fahrrädern     1351
Einfacher Diebstahl aus 

In [8]:
# inspecting the reported financial-loss field and its relationship with attempted cases
damage_summary = df["SCHADENSHOEHE"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

display(damage_summary.to_frame("SCHADENSHOEHE"))

print("Zero values:", (df["SCHADENSHOEHE"] == 0).sum())
print("Negative values:", (df["SCHADENSHOEHE"] < 0).sum())

damage_by_attempt = (
    df.groupby("VERSUCH")["SCHADENSHOEHE"]
    .agg(["count", "sum", "mean", "median", "min", "max"])
)

display(damage_by_attempt)

,SCHADENSHOEHE
count,26965.000000
mean,1271.269905
std,1200.297634
min,0.000000
25%,500.000000
50%,900.000000
75%,1618.000000
90%,2800.000000
95%,3699.000000
99%,5899.000000


Zero values: 234
Negative values: 0


,count,sum,mean,median,min,max
VERSUCH,,,,,,
Ja,154,0,0.000000,0.0,0,0
Nein,26804,34275044,1278.728697,900.0,0,10000
Unbekannt,7,4749,678.428571,300.0,0,2000


In [9]:
# checking the two cellar definitions
cellar_burglary_count = (
    df["DELIKT"] == "Keller- und Bodeneinbruch"
).sum()

cellar_location_count = (
    df["ERFASSUNGSGRUND"]
    .str.contains("Keller/Boden", na=False)
).sum()

print("Cellar burglary cases:", cellar_burglary_count)
print("Any cellar-location cases:", cellar_location_count)

Cellar burglary cases: 1351
Any cellar-location cases: 1421


### Reported financial-loss findings

- `SCHADENSHOEHE` represents the reported damage amount, not necessarily the bicycle’s purchase price.
- Values range from €0 to €10,000.
- The median is €900, while the mean is approximately €1,271.
- Because the mean is higher than the median, the distribution is influenced by high-value incidents.
- There are 234 records with €0 reported damage.
- All 154 attempted incidents have €0 reported damage.
- Another 80 completed or unknown-status incidents have €0 reported damage.
- No negative financial values were found.
- Attempted incidents should be analyzed separately from completed incidents.
- Both the median and mean should be reported.
- There are 1,351 cellar-burglary cases based on `DELIKT`.
- There are 1,421 cases connected to a cellar location based on `ERFASSUNGSGRUND`.
- Separate flags will be created for cellar burglary and cellar location.

In [10]:
# calculating the possible offence duration (This checks whether the reported time is exact or represents a broad window)
# Create start and end datetimes

df["offence_start"] = (
    df["TATZEIT_ANFANG_DATUM"]
    + pd.to_timedelta(df["TATZEIT_ANFANG_STUNDE"], unit="h")
)

df["offence_end"] = (
    df["TATZEIT_ENDE_DATUM"]
    + pd.to_timedelta(df["TATZEIT_ENDE_STUNDE"], unit="h")
)

# Calculate the possible offence window in hours

df["offence_window_hours"] = (
    df["offence_end"] - df["offence_start"]
).dt.total_seconds() / 3600

display(
    df["offence_window_hours"]
    .describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
    .to_frame("offence_window_hours")
)

print(
    "Negative durations:",
    (df["offence_window_hours"] < 0).sum()
)

print(
    "Zero-hour windows:",
    (df["offence_window_hours"] == 0).sum()
)

print(
    "Same-date cases:",
    (df["TATZEIT_ANFANG_DATUM"] == df["TATZEIT_ENDE_DATUM"]).sum()
)

print(
    "Windows longer than 24 hours:",
    (df["offence_window_hours"] > 24).sum()
)

,offence_window_hours
count,26965.000000
mean,13.874356
std,16.059363
min,0.000000
25%,2.000000
50%,9.000000
75%,18.000000
90%,39.000000
95%,51.000000
99%,69.000000


Negative durations: 0
Zero-hour windows: 3075
Same-date cases: 14348
Windows longer than 24 hours: 4329


### Offence-time reliability

- The dataset records the beginning and end of the possible offence period, not necessarily the exact theft moment.
- The median possible offence window is 9 hours.
- Only 3,075 incidents have a zero-hour window.
- 4,329 incidents have a possible window longer than 24 hours.
- The maximum possible window is 72 hours.
- No negative durations were detected.
- Hour-of-day analysis must therefore be treated cautiously.
- The project will either restrict detailed hourly analysis to narrow time windows or clearly label results as based on the beginning of the reported offence period.
- Month and broader weekday/weekend analyses are more reliable than claims about the exact theft hour.

In [11]:
# checking the number of records per year and create a fair comparison period
# Checking yearly coverage

df["year"] = df["TATZEIT_ANFANG_DATUM"].dt.year

print("All available records by year:")
print(df["year"].value_counts().sort_index())

# Comparing the same period in both years:
# 1 January through 8 August

period_2025 = df[
    df["TATZEIT_ANFANG_DATUM"].between(
        "2025-01-01",
        "2025-08-08"
    )
]

period_2026 = df[
    df["TATZEIT_ANFANG_DATUM"].between(
        "2026-01-01",
        "2026-08-08"
    )
]

print("\nComparable periods:")
print("2025-01-01 to 2025-08-08:", len(period_2025))
print("2026-01-01 to 2026-08-08:", len(period_2026))

All available records by year:
year
2025    17336
2026     9629
Name: count, dtype: int64

Comparable periods:
2025-01-01 to 2025-08-08: 11185
2026-01-01 to 2026-08-08: 9629


### Year coverage and comparison period

- The dataset contains 17,336 reports with a 2025 offence-start date.
- It contains 9,629 reports with a 2026 offence-start date through 8 August 2026.
- The complete 2025 total must not be compared directly with incomplete 2026.
- For year-over-year analysis, the project will compare matching periods:
  - 1 January–8 August 2025: 11,185 reports
  - 1 January–8 August 2026: 9,629 reports
- All charts comparing the two years must clearly state that matching year-to-date periods are being used.